In [3]:
# Standard library
import math
import warnings
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
from mplsoccer import Pitch

# Football data||
from statsbombpy import sb

# Machine learning
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Utilities
from tqdm import tqdm

warnings.filterwarnings("ignore", message="credentials were not supplied")

In [4]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Call statsbomb API to get all free competitions, then check Women's comps
free_comps = sb.competitions()
laliga_comps = free_comps[
    (free_comps['competition_gender'] == 'male') & 
    (free_comps['competition_name'] == 'La Liga')
]

laliga_comps

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
38,11,90,Spain,La Liga,male,False,False,2020/2021,2025-01-29T17:39:28.924386,2025-01-29T18:09:18.620699,2025-01-29T18:09:18.620699,2025-01-29T17:39:28.924386
39,11,42,Spain,La Liga,male,False,False,2019/2020,2024-12-16T16:51:06.833054,2021-06-13T16:17:31.694,None,2024-12-16T16:51:06.833054
40,11,4,Spain,La Liga,male,False,False,2018/2019,2024-09-22T18:50:23.364757,2021-07-09T14:53:22.103024,None,2024-09-22T18:50:23.364757
41,11,1,Spain,La Liga,male,False,False,2017/2018,2025-07-14T10:01:16.674864,2021-06-13T16:17:31.694,None,2025-07-14T10:01:16.674864
42,11,2,Spain,La Liga,male,False,False,2016/2017,2025-03-09T16:03:54.962718,2021-06-13T16:17:31.694,None,2025-03-09T16:03:54.962718
43,11,27,Spain,La Liga,male,False,False,2015/2016,2025-04-23T13:59:22.835792,2021-06-13T16:17:31.694,None,2025-04-23T13:59:22.835792
44,11,26,Spain,La Liga,male,False,False,2014/2015,2025-06-24T13:39:01.068680,2021-06-13T16:17:31.694,None,2025-06-24T13:39:01.068680
45,11,25,Spain,La Liga,male,False,False,2013/2014,2025-06-24T09:52:44.048585,2021-06-13T16:17:31.694,None,2025-06-24T09:52:44.048585
46,11,24,Spain,La Liga,male,False,False,2012/2013,2025-06-24T13:53:14.047130,2021-06-13T16:17:31.694,None,2025-06-24T13:53:14.047130
47,11,23,Spain,La Liga,male,False,False,2011/2012,2025-06-24T13:38:10.764259,2021-06-13T16:17:31.694,None,2025-06-24T13:38:10.764259


In [7]:
# Download Matches

leagues = list(zip(
    laliga_comps["competition_id"],
    laliga_comps["season_id"]
))

matches_df = pd.concat([
    sb.matches(competition_id=comp, season_id=season)
    for comp, season in leagues
], ignore_index=True)

len(matches_df)

868

In [8]:
matches_df.head()

,match_id,match_date,kick_off,competition,season,home_team,away_team,home_score,away_score,match_status,match_status_360,last_updated,last_updated_360,match_week,competition_stage,stadium,referee,home_managers,away_managers,data_version,shot_fidelity_version,xy_fidelity_version
0,3773386,2020-10-31,21:00:00.000,Spain - La Liga,2020/2021,Deportivo Alavés,Barcelona,1,1,available,available,2023-07-25T03:54:59.280826,2023-07-25T04:25:41.348202,8,Regular Season,Estadio de Mendizorroza,NaN,Pablo Javier Machín Díez,Ronald Koeman,1.1.0,2,2
1,3773565,2021-01-09,18:30:00.000,Spain - La Liga,2020/2021,Granada,Barcelona,0,4,available,available,2023-07-25T03:51:37.437064,2023-07-25T04:30:16.058384,18,Regular Season,Estadio Nuevo Los Cármenes,Ricardo De Burgos Bengoetxea,Diego Martínez Penas,Ronald Koeman,1.1.0,2,2
2,3773457,2021-05-16,18:30:00.000,Spain - La Liga,2020/2021,Barcelona,Celta Vigo,1,2,available,available,2022-12-02T09:26:39.496362,2023-04-27T23:03:53.506485,37,Regular Season,Spotify Camp Nou,NaN,Ronald Koeman,Eduardo Germán Coudet,1.1.0,2,2
3,3773631,2021-02-07,21:00:00.000,Spain - La Liga,2020/2021,Real Betis,Barcelona,2,3,available,available,2023-07-25T03:47:44.278651,2023-07-25T03:56:34.733180,22,Regular Season,Estadio Benito Villamarín,NaN,Manuel Luis Pellegrini Ripamonti,Ronald Koeman,1.1.0,2,2
4,3773665,2021-03-06,21:00:00.000,Spain - La Liga,2020/2021,Osasuna,Barcelona,0,2,available,available,2022-12-02T08:46:42.897589,2023-04-28T02:57:03.412841,26,Regular Season,Estadio El Sadar,Guillermo Cuadra Fernández,Jagoba Arrasate Elustondo,Ronald Koeman,1.1.0,2,2


In [5]:
sb.lineups(match_id=3773386)["Barcelona"]

,player_id,player_name,player_nickname,jersey_number,country,cards,positions
0,4447,Martin Braithwaite Christensen,Martin Braithwaite,9,Denmark,[],"[{'position_id': 21, 'position': 'Left Wing', ..."
1,5203,Sergio Busquets i Burgos,Sergio Busquets,5,Spain,"[{'time': '43:12', 'card_type': 'Yellow Card',...","[{'position_id': 9, 'position': 'Right Defensi..."
2,5211,Jordi Alba Ramos,Jordi Alba,18,Spain,[],"[{'position_id': 6, 'position': 'Left Back', '..."
3,5213,Gerard Piqué Bernabéu,Gerard Piqué,3,Spain,[],"[{'position_id': 3, 'position': 'Right Center ..."
4,5477,Ousmane Dembélé,None,11,France,[],"[{'position_id': 17, 'position': 'Right Wing',..."
5,5487,Antoine Griezmann,None,7,France,[],"[{'position_id': 23, 'position': 'Center Forwa..."
6,5503,Lionel Andrés Messi Cuccittini,Lionel Messi,10,Argentina,"[{'time': '39:05', 'card_type': 'Yellow Card',...","[{'position_id': 19, 'position': 'Center Attac..."
7,6379,Sergi Roberto Carnicer,Sergi Roberto,20,Spain,[],"[{'position_id': 2, 'position': 'Right Back', ..."
8,6590,Norberto Murara Neto,Neto,13,Brazil,[],"[{'position_id': 1, 'position': 'Goalkeeper', ..."
9,6826,Clément Lenglet,None,15,France,"[{'time': '43:20', 'card_type': 'Yellow Card',...","[{'position_id': 5, 'position': 'Left Center B..."


In [9]:
from statsbombpy import sb
import pandas as pd

def get_starting_xi(match_id: int, team: str) -> pd.DataFrame:
    """
    Extract the starting XI for a given team and match.
    """

    # Load lineup
    lineup = sb.lineups(match_id=match_id)[team].copy()

    # Expand positions (only needed to identify starters cleanly)
    df = lineup.explode("positions")
    pos = pd.json_normalize(df["positions"])

    df = pd.concat(
        [df.drop(columns="positions").reset_index(drop=True), pos],
        axis=1
    )

    # Identify starting XI
    starters = df[df["start_reason"] == "Starting XI"].copy()

    # Clean player names
    starters["player"] = (
        starters["player_nickname"]
        .fillna(starters["player_name"])
        .str.strip()
    )

    # Keep only relevant columns
    starters = starters[[
        "player",
        "jersey_number",
        "position"
    ]].sort_values("jersey_number")

    return starters.reset_index(drop=True)

In [10]:
match_id = 3773386
team = "Barcelona"

starting_xi = get_starting_xi(match_id, team)
print(starting_xi)

               player  jersey_number                   position
0        Gerard Piqué              3          Right Center Back
1     Sergio Busquets              5   Right Defensive Midfield
2   Antoine Griezmann              7             Center Forward
3        Lionel Messi             10  Center Attacking Midfield
4     Ousmane Dembélé             11                 Right Wing
5                Neto             13                 Goalkeeper
6     Clément Lenglet             15           Left Center Back
7          Jordi Alba             18                  Left Back
8       Sergi Roberto             20                 Right Back
9     Frenkie de Jong             21    Left Defensive Midfield
10          Ansu Fati             22                  Left Wing
